# 12 — Local-Level Values Clusters (K-Means per LA)

Repeats step 9's values-based K-Means independently for each Local Authority,
weighting each respondent by how many times they appear in that LA's
synthetic population (from step 10).

**Inputs:**

- `data/4_feature_eng/{WAVE}_feature_eng.pkl` — UKHLS feature table
- `data/10_synthetic_population/pidp_la_counts.csv` — `pidp`, `ladcd`, `ladnm`, `n`

**Output:** `data/12_cluster_values_LA/la_values_clusters.csv`
— one row per `(pidp × ladcd)` with columns: `pidp`, `ladcd`, `ladnm`, `n`, `cluster`

**Config** (`config_cluster.py`):

- `N_CLUSTERS_LOCAL` — clusters per LA (default 5)
- `TEST_MODE` / `TEST_LA_CODES` — limit to a subset of LAs for testing


In [8]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import data_pipeline.helpers.cluster as cf
importlib.reload(cf)
import data_pipeline.helpers.normalise as nf
importlib.reload(nf)

import data_pipeline.config_cluster as _cc
importlib.reload(_cc)
from data_pipeline.config_cluster import (
    N_CLUSTERS_LOCAL, TEST_MODE, TEST_LA_CODES, WAVE, HIERARCHICAL_CLUSTER
)

import numpy as np
import pandas as pd
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
FEATURE_PKL = Path(f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl")
LA_COUNTS   = Path(f"../{DATA_FOLDER}/10_synthetic_population/pidp_la_counts.csv")
OUT_DIR     = Path(f"../{DATA_FOLDER}/12_cluster_values_LA")
OUT_CSV     = OUT_DIR / "la_values_clusters.csv"

for p in (FEATURE_PKL, LA_COUNTS):
    if not p.exists():
        raise FileNotFoundError(f"{p} — check pipeline prerequisites.")

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Build hierarchical group label map ────────────────────────────────────────
hier_col    = None
hier_labels = {}
if HIERARCHICAL_CLUSTER:
    hier_col = f"{WAVE}_{HIERARCHICAL_CLUSTER}"
    base = HIERARCHICAL_CLUSTER.replace("_eng", "")   # "jbstat_eng" → "jbstat"
    vdef = _cv.VARIABLES.get(base, {})
    raw_map = vdef.get("group_labels") or vdef.get("categories") or {}
    hier_labels = {float(k): str(v) for k, v in raw_map.items()}
    print(f"Hierarchical clustering on '{hier_col}' — {len(hier_labels)} groups: "
          f"{list(hier_labels.values())}")

# ── Load features ─────────────────────────────────────────────────────────────
print("Loading feature table …")
df_features = pd.read_pickle(FEATURE_PKL)
df_features["pidp"] = pd.to_numeric(df_features["pidp"], errors="coerce").astype("int64")

if hier_col and hier_col not in df_features.columns:
    raise ValueError(f"HIERARCHICAL_CLUSTER column '{hier_col}' not found in feature table")

VARIABLES = _cv.VARIABLES
avail_feat_prefixed = [
    f"{WAVE}_{col}"
    for col in VARIABLES.keys()
    if f"{WAVE}_{col}" in df_features.columns
]
print(f"  {len(df_features):,} respondents, {len(avail_feat_prefixed)} feature columns")

# ── Normalise features ────────────────────────────────────────────────────────
print("Normalising features …")
coeffs   = nf.fit(df_features, avail_feat_prefixed)
df_normed = nf.apply(df_features, coeffs)
X_all    = df_normed[avail_feat_prefixed].values.astype(np.float32)
np.nan_to_num(X_all, copy=False, nan=0.0)

pidp_vals   = df_features["pidp"].values
pidp_to_row = {int(p): i for i, p in enumerate(pidp_vals)}

# pidp → group code lookup (for hierarchical mode)
pidp_to_group = {}
if hier_col:
    pidp_to_group = (
        pd.Series(df_features[hier_col].values, index=df_features["pidp"].values)
        .to_dict()
    )

# ── Load LA counts ────────────────────────────────────────────────────────────
print("Loading LA counts …")
df_la = pd.read_csv(LA_COUNTS, dtype={"pidp": "int64", "ladcd": str, "ladnm": str, "n": "int64"})
print(f"  {len(df_la):,} (pidp × LA) rows, {df_la['ladcd'].nunique():,} LAs")

if TEST_MODE and TEST_LA_CODES:
    df_la = df_la[df_la["ladcd"].isin(TEST_LA_CODES)].copy()
    print(f"  TEST MODE — restricted to {df_la['ladcd'].nunique()} LA(s): "
          f"{sorted(df_la['ladnm'].unique().tolist())}")

all_la_codes = sorted(df_la["ladcd"].unique())
mode_txt = f"hierarchical on '{HIERARCHICAL_CLUSTER}'" if HIERARCHICAL_CLUSTER else "flat"
print(f"  Processing {len(all_la_codes)} LA(s) with N_CLUSTERS_LOCAL={N_CLUSTERS_LOCAL} [{mode_txt}]")

# ── Cluster per LA ────────────────────────────────────────────────────────────
results = []

for i, ladcd in enumerate(all_la_codes, 1):
    la_rows = df_la[df_la["ladcd"] == ladcd].copy()
    ladnm   = la_rows["ladnm"].iloc[0]
    la_rows = la_rows[la_rows["pidp"].isin(pidp_to_row)].copy()

    if HIERARCHICAL_CLUSTER:
        la_rows["_group_code"] = la_rows["pidp"].map(pidp_to_group)
        group_codes = sorted(la_rows["_group_code"].dropna().unique())

        for gcode in group_codes:
            g_rows = la_rows[la_rows["_group_code"] == gcode].copy()
            g_n    = len(g_rows)
            glabel = hier_labels.get(gcode, str(int(gcode)))

            if g_n < N_CLUSTERS_LOCAL:
                print(f"  [{i}/{len(all_la_codes)}] {ladnm} / {glabel} "
                      f"— only {g_n} pidps, skipping")
                continue

            row_indices = g_rows["pidp"].map(pidp_to_row).values
            X_g = X_all[row_indices]
            w_g = g_rows["n"].values.astype(np.float64)
            k   = min(N_CLUSTERS_LOCAL, g_n)
            labels = cf.fit_kmeans(X_g, k, sample_weight=w_g)

            g_rows["cluster"] = labels + 1
            g_rows["group"]   = glabel
            results.append(g_rows[["pidp", "ladcd", "ladnm", "n", "cluster", "group"]])

            counts_str = ", ".join(
                f"{v}" for v in sorted(pd.Series(labels + 1).value_counts().sort_index().values)
            )
            print(f"  [{i}/{len(all_la_codes)}] {ladnm} / {glabel} "
                  f"— {g_n} pidps → {k} clusters [{counts_str}]")
    else:
        n_pidps = len(la_rows)
        if n_pidps < N_CLUSTERS_LOCAL:
            print(f"  [{i}/{len(all_la_codes)}] {ladnm} — only {n_pidps} pidps, skipping")
            continue

        row_indices = la_rows["pidp"].map(pidp_to_row).values
        X_la = X_all[row_indices]
        w_la = la_rows["n"].values.astype(np.float64)
        k    = min(N_CLUSTERS_LOCAL, n_pidps)
        labels = cf.fit_kmeans(X_la, k, sample_weight=w_la)

        la_rows["cluster"] = labels + 1
        results.append(la_rows)

        counts_str = ", ".join(
            f"{v}" for v in sorted(pd.Series(labels + 1).value_counts().sort_index().values)
        )
        print(f"  [{i}/{len(all_la_codes)}] {ladnm} — {n_pidps} pidps → {k} clusters [{counts_str}]")

# ── Save raw pidp-level output ────────────────────────────────────────────────
df_out = pd.concat(results, ignore_index=True)
df_out.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(df_out):,} rows → {OUT_CSV}")

# ── Build demographic summary per cluster ─────────────────────────────────────
print("\nBuilding demographic summary …")
import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)

df_merged = df_out.rename(columns={"n": "n_sipher_rows"}).merge(df_features, on="pidp", how="left")

summary_rows = []
if HIERARCHICAL_CLUSTER:
    for (ladcd_val, group_val), grp in df_merged.groupby(["ladcd", "group"]):
        la_summary = _cs.make_cluster_summary(grp, "cluster", wave=WAVE)
        la_summary["ladcd"] = ladcd_val
        la_summary["ladnm"] = grp["ladnm"].iloc[0]
        la_summary["group"] = group_val
        summary_rows.append(la_summary)
else:
    for ladcd_val, grp in df_merged.groupby("ladcd"):
        la_summary = _cs.make_cluster_summary(grp, "cluster", wave=WAVE)
        la_summary["ladcd"] = ladcd_val
        la_summary["ladnm"] = grp["ladnm"].iloc[0]
        summary_rows.append(la_summary)

df_summary = pd.concat(summary_rows, ignore_index=True)

leading = ["ladcd", "ladnm"] + (["group"] if HIERARCHICAL_CLUSTER else [])
df_summary = df_summary[leading + [c for c in df_summary.columns if c not in leading]]

# ── Copy summary to API folder ────────────────────────────────────────────────
_api_dir  = Path("../api/data/clusters")
_api_dir.mkdir(parents=True, exist_ok=True)
_api_dest = _api_dir / "local_values_clusters.csv"
df_summary.to_csv(_api_dest, index=False)
print(f"Summary ({len(df_summary)} rows) → {_api_dest}")

preview_cols = leading + ["cluster_id", "tribe_label", "n_respondents", "size"]
print(df_summary[[c for c in preview_cols if c in df_summary.columns]].to_string(index=False))


Hierarchical clustering on 'k_jbstat_eng' — 7 groups: ['Not provided', 'Employed', 'Unemployed', 'Retired', 'On leave', 'Student / training', 'Inactive']
Loading feature table …
  27,330 respondents, 9 feature columns
Normalising features …
Loading LA counts …
  7,969,122 (pidp × LA) rows, 346 LAs
  TEST MODE — restricted to 6 LA(s): ['Blackpool', 'County Durham', 'Hounslow', 'Islington', 'Newham', 'Tower Hamlets']
  Processing 6 LA(s) with N_CLUSTERS_LOCAL=5 [hierarchical on 'jbstat_eng']
  [1/6] Blackpool / Employed — 11511 pidps → 5 clusters [310, 1324, 1617, 4108, 4152]
  [1/6] Blackpool / Unemployed — 739 pidps → 5 clusters [59, 90, 96, 134, 360]
  [1/6] Blackpool / Retired — 5484 pidps → 5 clusters [231, 377, 933, 1894, 2049]
  [1/6] Blackpool / On leave — 739 pidps → 5 clusters [45, 93, 96, 173, 332]
  [1/6] Blackpool / Student / training — 916 pidps → 5 clusters [83, 85, 91, 318, 339]
  [1/6] Blackpool / Inactive — 826 pidps → 5 clusters [8, 96, 109, 123, 490]
  [2/6] County Du

In [ ]:
# ── LLM Labelling ─────────────────────────────────────────────────────────────
import importlib
import data_pipeline.helpers.llm_prompts as _lp;  importlib.reload(_lp)
import data_pipeline.helpers.llm_gemini as _lgemini;  importlib.reload(_lgemini)
from dotenv import load_dotenv
import os, json, re, time

load_dotenv(dotenv_path=Path("..") / ".env")
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
LABEL_MODEL    = "gemini-2.5-flash"
LABEL_DELAY    = 5   # seconds between calls (rate-limit headroom)

raw_chat = _lgemini.make_gemini_caller(GOOGLE_API_KEY, LABEL_MODEL)

def _extract_json(text: str) -> dict:
    clean = re.sub(r"```(?:json)?", "", text).strip(" `")
    return json.loads(clean)

group_cols = ["ladcd", "group"] if "group" in df_summary.columns else ["ladcd"]
print(f"Labelling clusters with {LABEL_MODEL} ({len(df_summary.groupby(group_cols))} LA×group batches) …")

for keys, grp in df_summary.groupby(group_cols):
    ladcd_val = keys[0] if isinstance(keys, tuple) else keys
    group_val = keys[1] if isinstance(keys, tuple) and len(keys) > 1 else None
    la_name   = grp["ladnm"].iloc[0]
    emp_label = _lp.resolve_emp_label(group_val) if group_val is not None else None
    group_disp = emp_label or str(group_val) if group_val is not None else None

    print(f"  {la_name} / {group_disp or 'all'} — {len(grp)} clusters …", end=" ", flush=True)

    user_prompt = _lp.build_label_prompt(grp, group_disp or "", la_name, emp_label)
    response    = raw_chat(_lp.LABEL_SYSTEM_PROMPT, user_prompt, temperature=0.3)

    try:
        payload = _extract_json(response)
        labels  = payload["labels"]
        if len(labels) != len(grp):
            raise ValueError(f"Got {len(labels)} labels for {len(grp)} clusters")
    except Exception as e:
        print(f"PARSE ERROR: {e}  — keeping default labels")
        time.sleep(LABEL_DELAY)
        continue

    for (idx, _row), label in zip(grp.iterrows(), labels):
        df_summary.at[idx, "tribe_label"] = _lp.sanitise_name(label)

    print("OK")
    time.sleep(LABEL_DELAY)

# Re-save with updated labels
df_summary.to_csv(_api_dest, index=False)
print(f"\nRe-saved with LLM labels → {_api_dest}")
preview_cols = ["ladcd", "ladnm"] + (["group"] if "group" in df_summary.columns else []) + ["cluster_id", "tribe_label", "size"]
print(df_summary[[c for c in preview_cols if c in df_summary.columns]].to_string(index=False))
